# A2A MCP Client

Test client for the orchestrator agent. Run all agent notebooks first:
1. `mcp_server.ipynb` (port 10100)
2. `planner_agent.ipynb` (port 10102)
3. `travel_agents.ipynb` (ports 10103-10105)
4. `orchestrator_agent.ipynb` (port 10101)

In [ ]:
import json

from a2a.client import ClientFactory
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest,
    TaskArtifactUpdateEvent,
    TaskStatusUpdateEvent,
)
from google.protobuf import json_format

ORCHESTRATOR_URL = 'http://localhost:10101'

## Fetch Agent Card

In [ ]:
import httpx

resp = httpx.get(f'{ORCHESTRATOR_URL}/.well-known/agent-card.json')
resp.raise_for_status()
print(json.dumps(resp.json(), indent=2))

## Send Streaming Message

In [ ]:
async def send_message(text: str, task_id: str = None, context_id: str = None):
    """Send a message to the orchestrator and print streaming events."""
    from uuid import uuid4
    client = await ClientFactory.connect(ORCHESTRATOR_URL)
    message = Message(
        role=Role.ROLE_USER,
        parts=[Part(text=text)],
        message_id=uuid4().hex,
    )
    if task_id:
        message.task_id = task_id
    if context_id:
        message.context_id = context_id

    request = SendMessageRequest(message=message)
    async for stream_resp, task in client.send_message(request):
        payload_type = stream_resp.WhichOneof('payload')
        if payload_type == 'status_update':
            evt = stream_resp.status_update
            msg_text = evt.status.message.parts[0].text if evt.status.message.parts else ''
            print(f'[{TaskState.Name(evt.status.state)}] {msg_text}')
            task_id = evt.task_id
            context_id = evt.context_id
        elif payload_type == 'artifact_update':
            artifact = stream_resp.artifact_update.artifact
            print(f'[artifact] {artifact.name}')
            for part in artifact.parts:
                content_type = part.WhichOneof('content')
                if content_type == 'text':
                    print(part.text)
                elif content_type == 'data':
                    print(json.dumps(json_format.MessageToDict(part.data), indent=2))
    return task_id, context_id

In [ ]:
task_id, context_id = await send_message('Plan my trip to London from San Francisco')

In [ ]:
# Continue the conversation if the agent needs more input
# task_id, context_id = await send_message('Budget is $5000, business trip, 1 traveler, business class', task_id, context_id)